# 02. XGBoost Training (v2) — Refactored for AUROC/AUPRC

- 환자 단위 group split을 전제로 `train.csv/val.csv/test.csv`를 사용합니다.
- 평가/early stopping은 **AUC-PR(aucpr)** 중심으로 최적화합니다.
- 타겟별 `scale_pos_weight`를 적용합니다.


In [ ]:
# ==============================================================================
# Imports
# ==============================================================================
import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

warnings.filterwarnings("ignore")

print("=== 02. XGBoost Training 시작 (Refactored) ===")
print("xgboost:", xgb.__version__)


In [ ]:
# ==============================================================================
# Config
# ==============================================================================
DATA_DIR = "../../data-pipeline/data/processed/all_features_refactored"   # 01_model_prep 결과 폴더
OUTPUT_DIR = "../models/xgboost"
VERSION = "bv1"

TARGETS = {
    "death": "death_next_24h",
    "vent": "vent_next_12h",
    "pressor": "pressor_next_12h",
    "composite": "composite_next_24h",
}

# 임계값(운영용) — 점수(AUROC/AUPRC)에는 영향이 없고, 분류지표에만 영향
DEFAULT_THRESHOLDS = {
    "death": 0.10,
    "vent": 0.10,
    "pressor": 0.10,
    "composite": 0.15,
}

os.makedirs(os.path.join(OUTPUT_DIR, VERSION), exist_ok=True)
print("DATA_DIR:", os.path.abspath(DATA_DIR))
print("OUTPUT_DIR:", os.path.abspath(os.path.join(OUTPUT_DIR, VERSION)))


In [ ]:
# ==============================================================================
# Load data (prep 단계에서 이미 group split 된 train/val/test)
# ==============================================================================
train_path = os.path.join(DATA_DIR, "train.csv")
val_path   = os.path.join(DATA_DIR, "val.csv")
test_path  = os.path.join(DATA_DIR, "test.csv")
feat_path  = os.path.join(DATA_DIR, "features.json")
w_path     = os.path.join(DATA_DIR, "scale_pos_weight.json")

train = pd.read_csv(train_path)
val   = pd.read_csv(val_path)
test  = pd.read_csv(test_path)

with open(feat_path, "r") as f:
    feature_cols = json.load(f)

with open(w_path, "r") as f:
    scale_pos_weights = json.load(f)

print(f"Train: {train.shape} | Val: {val.shape} | Test: {test.shape}")
print(f"Features: {len(feature_cols)}")
print("Targets:", TARGETS)


In [ ]:
# ==============================================================================
# Quick sanity checks (누수 방지 가정 검증용)
# ==============================================================================
# (1) 타겟 컬럼 존재
missing_targets = [c for c in TARGETS.values() if c not in train.columns]
assert not missing_targets, f"Missing target columns: {missing_targets}"

# (2) 피처 컬럼 존재
missing_feats = [c for c in feature_cols if c not in train.columns]
assert not missing_feats, f"Missing feature columns: {missing_feats}"

# (3) 분할별 양성비 확인 (AUPRC가 prevalence에 민감)
for split_name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"\n[{split_name.upper()}] label prevalence")
    for k, col in TARGETS.items():
        pr = df[col].mean()
        print(f"  {k:<10} {col:<22} pos_rate={pr:.5f} (pos={int(df[col].sum()):,}/{len(df):,})")


## 학습 전략 (AUROC/AUPRC 올리기)

- `eval_metric=['auc','aucpr']`로 **랭킹 품질**을 직접 모니터링
- `early_stopping_rounds`는 **aucpr** 개선이 멈출 때 기준
- 타겟별 `scale_pos_weight` 적용
- (옵션) 작은 랜덤/그리드 서치로 **val AUPRC**가 가장 좋은 파라미터 선택


In [ ]:
# ==============================================================================
# Parameter search space (작게, 하지만 효과 큰 축만)
# - 너무 큰 탐색은 노트북에서 시간 오래 걸릴 수 있어, 기본은 '작은 후보 중 선택'
# ==============================================================================
BASE_PARAMS = dict(
    objective="binary:logistic",
    tree_method="hist",
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=0.0,
    # NOTE: xgboost>=1.6: eval_metric을 params에 넣거나 train()에 넣어도 됨
    eval_metric=["auc", "aucpr"],
    random_state=42,
)

SEARCH_SPACE = [
    # (max_depth, min_child_weight, subsample, colsample, reg_lambda)
    (3, 20, 0.8, 0.8, 3.0),
    (4, 10, 0.8, 0.8, 2.0),
    (4, 20, 0.8, 0.8, 3.0),
    (5, 10, 0.8, 0.8, 2.0),
    (3, 10, 0.7, 0.7, 2.0),
    (4, 10, 0.9, 0.9, 2.0),
]

NUM_BOOST_ROUND = 5000
EARLY_STOPPING_ROUNDS = 200
VERBOSE_EVAL = 200

def make_dmatrix(df: pd.DataFrame, y: pd.Series):
    X = df[feature_cols]
    return xgb.DMatrix(X, label=y, feature_names=feature_cols)

dtrain_base = train[feature_cols]
dval_base   = val[feature_cols]
dtest_base  = test[feature_cols]

print("Search candidates:", len(SEARCH_SPACE))


In [ ]:
# ==============================================================================
# Train per target (select params by best VAL AUPRC)
# ==============================================================================
models = {}
best_params = {}
training_summary = {}

for name, target_col in TARGETS.items():
    print("\n" + "="*80)
    print(f"[{name.upper()}] target = {target_col}")
    print("="*80)

    y_tr = train[target_col].astype(int)
    y_va = val[target_col].astype(int)

    # scale_pos_weight (train 기준)
    spw = float(scale_pos_weights.get(target_col, 1.0))

    dtrain = xgb.DMatrix(train[feature_cols], label=y_tr, feature_names=feature_cols)
    dval   = xgb.DMatrix(val[feature_cols],   label=y_va, feature_names=feature_cols)

    best_aucpr = -1.0
    best = None
    best_bst = None

    for (md, mcw, ss, cs, rl) in SEARCH_SPACE:
        params = dict(BASE_PARAMS)
        params.update({
            "max_depth": md,
            "min_child_weight": mcw,
            "subsample": ss,
            "colsample_bytree": cs,
            "reg_lambda": rl,
            "scale_pos_weight": spw,
        })

        evals_result = {}
        bst = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=NUM_BOOST_ROUND,
            evals=[(dtrain, "train"), (dval, "val")],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            evals_result=evals_result,
            verbose_eval=False,
        )

        # best_score는 early stop 기준 metric(보통 마지막 eval_metric)로 잡힐 수 있어 직접 계산
        val_prob = bst.predict(dval, iteration_range=(0, bst.best_iteration + 1))
        aucpr = average_precision_score(y_va, val_prob)
        auroc = roc_auc_score(y_va, val_prob)

        if aucpr > best_aucpr:
            best_aucpr = aucpr
            best = params
            best_bst = bst
            best_metrics = (auroc, aucpr, bst.best_iteration)

    models[name] = best_bst
    best_params[name] = best
    training_summary[name] = {
        "val_auroc": float(best_metrics[0]),
        "val_auprc": float(best_metrics[1]),
        "best_iteration": int(best_metrics[2]),
        "scale_pos_weight": spw,
    }

    print(f"✓ Best VAL AUPRC: {best_metrics[1]:.6f} | VAL AUROC: {best_metrics[0]:.6f} | best_iter: {best_metrics[2]}")
    print(f"  params: max_depth={best['max_depth']} min_child_weight={best['min_child_weight']} subsample={best['subsample']} colsample={best['colsample_bytree']} reg_lambda={best['reg_lambda']} spw={spw}")

print("\nDone.")


In [ ]:
# ==============================================================================
# Evaluate on TEST
# ==============================================================================
results = {}
y_true_dict, y_prob_dict = {}, {}

for name, target_col in TARGETS.items():
    y_te = test[target_col].astype(int)
    dtest = xgb.DMatrix(test[feature_cols], label=y_te, feature_names=feature_cols)

    bst = models[name]
    prob = bst.predict(dtest, iteration_range=(0, bst.best_iteration + 1))

    auroc = roc_auc_score(y_te, prob)
    auprc = average_precision_score(y_te, prob)

    # 운영용 임계값으로 분류지표도 같이 출력(참고용)
    thr = float(DEFAULT_THRESHOLDS.get(name, 0.5))
    pred = (prob >= thr).astype(int)
    prec = precision_score(y_te, pred, zero_division=0)
    rec  = recall_score(y_te, pred, zero_division=0)
    f1   = f1_score(y_te, pred, zero_division=0)
    cm   = confusion_matrix(y_te, pred)

    results[name] = {
        "test_auroc": float(auroc),
        "test_auprc": float(auprc),
        "threshold": thr,
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "confusion_matrix": cm.tolist(),
        "pred_pos": int(pred.sum()),
        "actual_pos": int(y_te.sum()),
    }

    y_true_dict[name] = y_te
    y_prob_dict[name] = prob

print(json.dumps(results, indent=2))


In [ ]:
# ==============================================================================
# Threshold sweep (원하면 운영 정책에 맞는 threshold 선택)
# - AUROC/AUPRC는 threshold와 무관하지만, 알람 폭/Recall trade-off를 보기 좋게 출력
# ==============================================================================
THRESH_SWEEP = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

for name, target_col in TARGETS.items():
    y = y_true_dict[name]
    p = y_prob_dict[name]
    print("\n" + "="*70)
    print(f"[{name.upper()}] Threshold sweep")
    print("="*70)
    print(f"{'Threshold':>9}  {'Precision':>9}  {'Recall':>8}  {'F1':>8}  {'PredPos':>10}  {'ActualPos':>10}")
    for t in THRESH_SWEEP:
        pred = (p >= t).astype(int)
        prec = precision_score(y, pred, zero_division=0)
        rec  = recall_score(y, pred, zero_division=0)
        f1   = f1_score(y, pred, zero_division=0)
        print(f"{t:9.2f}  {prec:9.6f}  {rec:8.6f}  {f1:8.6f}  {int(pred.sum()):10,d}  {int(y.sum()):10,d}")


In [ ]:
# ==============================================================================
# Save models + configs
# ==============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = os.path.join(OUTPUT_DIR, VERSION, timestamp)
os.makedirs(out_dir, exist_ok=True)

# save models
for name, bst in models.items():
    model_path = os.path.join(out_dir, f"xgb_{name}.json")
    bst.save_model(model_path)

# save config
artifact = {
    "version": VERSION,
    "timestamp": timestamp,
    "data_dir": os.path.abspath(DATA_DIR),
    "feature_cols_path": os.path.abspath(feat_path),
    "n_features": len(feature_cols),
    "targets": TARGETS,
    "default_thresholds": DEFAULT_THRESHOLDS,
    "training_summary": training_summary,
    "best_params": best_params,
    "test_results": results,
}

with open(os.path.join(out_dir, "training_artifacts.json"), "w") as f:
    json.dump(artifact, f, indent=2)

print("✓ Saved to:", out_dir)
